# Phase 3B: Reusable EEG Preprocessing Pipeline & Post-Filter QC Screening

This notebook executes the **Phase 3B Reusable EEG Preprocessing Pipeline** (`imagined_speech.data.preprocessing`) across all 690 EDF recordings discovered in Phase 1.

### Approved Phase 3B Decisions & Scientific Safeguards
1. **Canonical 14-Channel EEG Montage**: Selects `AF3, F7, F3, FC5, T7, P7, O1, O2, P8, T8, FC6, F4, F8, AF4` in the established order and excludes auxiliary channels from EEG signal arrays while recording excluded channel names in metadata.
2. **Signal Conditioning**: Applies per-channel mean subtraction followed by a 4th-order Butterworth `0.5-45.0 Hz` zero-phase band-pass filter at `128 Hz` (`sosfiltfilt`).
3. **Padding & Boundary Validation**: Retains symmetric reflection padding (`padtype='even'`, target 3 cycles at `0.5 Hz` = `768 samples` = `6.0 s` at `128 Hz`), explicitly validates boundary vs. interior signal behavior, and records complete padding provenance.
4. **Reference Mode**: Defaults to `reference_mode: none` (no additional spatial rereferencing). Common Average Reference (`reference_mode: car`) remains a configurable experimental option and is not applied by default.
5. **Exploratory Post-Filter QC Screening Flags**: Computes post-filter channel-level QC metrics and preserves exploratory screening flags (`post_filter_std_uV < 0.5 uV` and `post_filter_ptp_uV > 1000.0 uV`), labeled strictly as screening flags rather than verified artifacts.
6. **100% Recording & Channel Retention**: Preserves all 690 recordings and all 14 EEG channels per recording without automatic rejection, channel removal, interpolation, or sample replacement.
7. **Strict Preservation of Source & Earlier-Phase Files**: Raw `.edf` files, Phase 1 (`phase1_audit/`), Phase 2 (`phase2_signal_quality/`), and Phase 3A (`phase3a_diagnostic/`) outputs remain byte-for-byte unchanged. Phase 3B outputs are written strictly to `/content/drive/MyDrive/imagined_speech_outputs/phase3b_preprocessing/` in Colab (`./outputs/phase3b_preprocessing/` locally).
8. **Scope Safeguards**: No imagined-speech onset/offset sub-intervals are inferred, filename prefixes remain `unverified_filename_prefix`, and no training windows, data splits, cross-recording normalization, feature extraction, or model training are performed in this phase.

## 1. Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository

In [ ]:
import subprocess

REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f'Cloning {REPO_URL} into {COLAB_REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO_DIR)], check=True)
    elif (COLAB_REPO_DIR / '.git').is_dir():
        print(f'Updating existing repository at {COLAB_REPO_DIR}...')
        subprocess.run(['git', '-C', str(COLAB_REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        raise RuntimeError(
            f'Directory {COLAB_REPO_DIR} exists but is not a valid Git repository.'
        )
    os.chdir(COLAB_REPO_DIR)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'pyproject.toml not found in working directory {REPO_ROOT}. '
        'Ensure the notebook is running from the repository root.'
    )

print(f'Repository root: {REPO_ROOT}')

## 3. Install the Package

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

src_dir = str((REPO_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import imagined_speech
print(f'Installed imagined_speech version: {imagined_speech.__version__}')

## 4. Verify Existing Phase 1 / Phase 2 / Phase 3A Reports and Snapshot SHA-256 Checksums
Confirm that Phase 1 (`manifest.csv`, `channel_audit.csv`, `audit_summary.json`) exists and snapshot SHA-256 checksums across `phase1_audit/`, `phase2_signal_quality/`, and `phase3a_diagnostic/` so we can verify at the end of the run that none of them changed.

In [ ]:
import hashlib
import json
import shutil
import tempfile
import pandas as pd
from IPython.display import display
from imagined_speech.data.signal_quality import load_and_validate_manifest

PHASE1_AUDIT_DIR = Path(
    os.environ.get(
        'ISR_PHASE1_AUDIT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase1_audit'
        if IN_COLAB
        else './outputs/phase1_audit',
    )
)
PHASE2_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE2_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase2_signal_quality'
        if IN_COLAB
        else './outputs/phase2_signal_quality',
    )
)
PHASE3A_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_PHASE3A_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3a_diagnostic'
        if IN_COLAB
        else './outputs/phase3a_diagnostic',
    )
)
PHASE3B_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase3b_preprocessing'
        if IN_COLAB
        else './outputs/phase3b_preprocessing',
    )
)
CONFIG_PATH = REPO_ROOT / 'configs' / 'default_config.yaml'


def snapshot_dir_hashes(directory: Path) -> dict[str, str]:
    if not directory.is_dir():
        return {}
    hashes: dict[str, str] = {}
    for fpath in sorted(p for p in directory.rglob('*') if p.is_file()):
        rel = fpath.relative_to(directory).as_posix()
        hashes[rel] = hashlib.sha256(fpath.read_bytes()).hexdigest()
    return hashes


REQUIRED_PHASE1_FILES = ('manifest.csv', 'channel_audit.csv', 'audit_summary.json')
missing_phase1 = [
    name
    for name in REQUIRED_PHASE1_FILES
    if not (PHASE1_AUDIT_DIR / name).is_file() or (PHASE1_AUDIT_DIR / name).stat().st_size == 0
]
if missing_phase1:
    raise FileNotFoundError(
        f'Required Phase 1 audit reports are missing or empty in {PHASE1_AUDIT_DIR}: '
        f'{missing_phase1}. Please run notebooks/01_phase1_dataset_audit.ipynb first.'
    )

phase1_hashes_before = snapshot_dir_hashes(PHASE1_AUDIT_DIR)
phase2_hashes_before = snapshot_dir_hashes(PHASE2_OUTPUT_DIR)
phase3a_hashes_before = snapshot_dir_hashes(PHASE3A_OUTPUT_DIR)

manifest_df = load_and_validate_manifest(PHASE1_AUDIT_DIR / 'manifest.csv')
print(f'Verified Phase 1 audit directory:  {PHASE1_AUDIT_DIR} ({len(phase1_hashes_before)} files snapshotted)')
print(f'Phase 2 output files snapshotted:  {len(phase2_hashes_before)}')
print(f'Phase 3A output files snapshotted: {len(phase3a_hashes_before)}')
print(f'Phase 3B target output directory:  {PHASE3B_OUTPUT_DIR}')
print(f'Loaded manifest_df with {len(manifest_df)} recording rows.')

## 5. Verify or Extract Dataset Archive, Resolve `DATA_ROOT`, and Snapshot Raw EDF Checksums

In [ ]:
import zipfile
from imagined_speech.exceptions import DatasetDiscoveryError

DRIVE_ZIP_PATH = Path(
    os.environ.get(
        'ISR_DRIVE_ZIP',
        '/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip',
    )
)
EXTRACT_ROOT = Path(
    os.environ.get(
        'ISR_EXTRACT_ROOT',
        '/content/data/raw' if IN_COLAB else './data/raw',
    )
)
CONFIRMED_COLAB_DATA_ROOT = Path('/content/data/raw/Imagined_speech_EEG_edf')
EXPECTED_TASKS = ('Char', 'Digit', 'Image')


def _has_expected_task_dirs(candidate: Path, expected_tasks: tuple[str, ...] = EXPECTED_TASKS) -> tuple[bool, list[str]]:
    if not candidate.is_dir():
        return False, []
    child_dirs = {
        child.name.lower(): child.name
        for child in sorted(candidate.iterdir(), key=lambda p: p.name.lower())
        if child.is_dir()
    }
    expected_lower = {t.lower() for t in expected_tasks}
    if expected_lower.issubset(child_dirs.keys()):
        return True, [child_dirs[t.lower()] for t in expected_tasks]
    return False, []


def locate_dataset_root(
    search_root: Path,
    expected_tasks: tuple[str, ...] = EXPECTED_TASKS,
) -> tuple[Path, list[str]]:
    if not search_root.exists() or not search_root.is_dir():
        raise DatasetDiscoveryError(
            f'Extraction directory does not exist or is not a directory: {search_root}'
        )
    candidate_dirs = [search_root] + sorted(
        (
            d
            for d in search_root.rglob('*')
            if d.is_dir() and not any(part.startswith('.') or part == '__MACOSX' for part in d.parts)
        ),
        key=lambda p: (len(p.parts), p.as_posix().lower()),
    )
    for cand in candidate_dirs:
        matched, folders = _has_expected_task_dirs(cand, expected_tasks)
        if matched:
            return cand.resolve(), folders
    raise DatasetDiscoveryError(
        f'Could not find a directory under {search_root} containing required '
        f'category folders {list(expected_tasks)}.'
    )


existing_edfs = (
    [p for p in EXTRACT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() == '.edf']
    if EXTRACT_ROOT.exists()
    else []
)
if existing_edfs:
    print(f'Skipping ZIP extraction: found {len(existing_edfs)} existing .edf files under {EXTRACT_ROOT}.')
else:
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(f'Dataset ZIP archive not found at {DRIVE_ZIP_PATH}.')
    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {DRIVE_ZIP_PATH} -> {EXTRACT_ROOT} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)

env_data_root = os.environ.get('ISR_DATA_ROOT')
if env_data_root:
    DATA_ROOT, discovered_categories = locate_dataset_root(Path(env_data_root))
elif IN_COLAB and _has_expected_task_dirs(CONFIRMED_COLAB_DATA_ROOT)[0]:
    DATA_ROOT = CONFIRMED_COLAB_DATA_ROOT.resolve()
    discovered_categories = _has_expected_task_dirs(CONFIRMED_COLAB_DATA_ROOT)[1]
else:
    DATA_ROOT, discovered_categories = locate_dataset_root(EXTRACT_ROOT)

raw_edf_hashes_before = snapshot_dir_hashes(DATA_ROOT)
print(f'Resolved DATA_ROOT:          {DATA_ROOT}')
print(f'Discovered category folders: {discovered_categories}')
print(f'Snapshotted raw EDF files:   {len(raw_edf_hashes_before)}')

## 6. Run Unit Tests and Phase 3B Smoke Test (1 Recording per Task)
Before processing all 690 recordings, run the unit test suite (`tests/test_preprocessing.py`) and execute a 3-recording smoke test (one from each of `Char`, `Digit`, and `Image`) into a temporary smoke directory. Validate the generated `.npz` signal archives, shapes, canonical channel order, padding metadata, boundary behavior, and QC screening records.

In [ ]:
from imagined_speech.config import DEFAULT_PRIMARY_EEG_MONTAGE, load_config
from imagined_speech.data.preprocessing import (
    load_preprocessed_recording,
    run_phase3b_preprocessing,
    verify_phase3b_outputs,
)
from imagined_speech.data.signal_quality import select_representative_recordings
from imagined_speech.utils import set_global_seed, setup_logging

print('Running unit tests for Phase 3B preprocessing pipeline...')
pytest_proc = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_preprocessing.py', '-v'],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
    check=False,
)
if pytest_proc.stdout:
    print(pytest_proc.stdout)
if pytest_proc.stderr:
    print(pytest_proc.stderr, file=sys.stderr)
if pytest_proc.returncode != 0:
    raise RuntimeError(
        f'Phase 3B unit tests failed with exit code {pytest_proc.returncode}. '
        'See captured pytest output above.'
    )

config = load_config(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    output_dir=PHASE3B_OUTPUT_DIR,
)
setup_logging(config.log_level)
set_global_seed(config.seed)

smoke_inputs = select_representative_recordings(
    manifest_df=manifest_df,
    data_root=config.data_root,
    expected_tasks=config.discovery.expected_tasks,
    recordings_per_task=1,
    specific_recordings=(),
    seed=config.seed,
)
smoke_paths = [s.relative_source_path for s in smoke_inputs]
print(f'Running Phase 3B smoke test on {len(smoke_paths)} recordings (1 per task): {smoke_paths}')

with tempfile.TemporaryDirectory(prefix='phase3b_smoke_') as smoke_tmp:
    smoke_dir = Path(smoke_tmp)
    smoke_result = run_phase3b_preprocessing(
        config=config,
        manifest_source=manifest_df,
        output_dir=smoke_dir,
        selected_relative_paths=smoke_paths,
        save_outputs=True,
        verify_completeness=True,
    )
    assert smoke_result.summary['n_recordings_processed'] == 3
    assert smoke_result.summary['n_channel_traces_processed'] == 42
    assert smoke_result.summary['output_verification']['all_outputs_verified'] is True

    sample_npz = Path(smoke_result.generated_npz_files[0])
    loaded_sample = load_preprocessed_recording(sample_npz)
    assert loaded_sample.channel_names == DEFAULT_PRIMARY_EEG_MONTAGE
    assert loaded_sample.eeg_signals_uV.shape[0] == 14
    assert loaded_sample.sfreq_hz == 128.0
    print(
        f'Validated smoke NPZ archive: {sample_npz.name} | shape={loaded_sample.eeg_signals_uV.shape} '
        f'| sfreq={loaded_sample.sfreq_hz} Hz | channels={loaded_sample.channel_names}'
    )

display(
    smoke_result.manifest_df[
        [
            'task_category',
            'relative_source_path',
            'normalized_label',
            'sfreq_hz',
            'n_samples',
            'duration_sec',
            'effective_padlen_samples',
            'reference_mode',
            'n_screening_flagged_channels',
            'recording_retained_in_outputs',
        ]
    ]
)
display(
    smoke_result.channel_qc_df[
        [
            'relative_source_path',
            'channel_name',
            'subtracted_dc_mean_uV',
            'pre_filter_std_uV',
            'post_filter_std_uV',
            'post_filter_ptp_uV',
            'post_filter_boundary_to_interior_std_ratio',
            'screening_flag_low_std',
            'screening_flag_high_ptp',
            'screening_status',
        ]
    ].head(14)
)

## 7. Run Full Phase 3B Preprocessing Pipeline Across All 690 Recordings
Process all 690 recordings and save the compressed per-recording `.npz` archives (`signals/<Task>/<File>.npz`), `preprocessed_manifest.csv`, `channel_qc_metrics.csv`, `phase3b_preprocessing_summary.json`, `phase3b_preprocessing_report.md`, and summary figures to `PHASE3B_OUTPUT_DIR`.

In [ ]:
phase3b_result = run_phase3b_preprocessing(
    config=config,
    manifest_source=manifest_df,
    output_dir=PHASE3B_OUTPUT_DIR,
    save_outputs=True,
    verify_completeness=True,
)
print('=== Phase 3B Preprocessing Summary (All 690 Recordings) ===')
print(json.dumps(phase3b_result.summary, indent=2))

## 8. Review Preprocessed Manifest, Channel QC Metrics, Boundary Validation, and Exploratory Screening Flags

In [ ]:
p_manifest = phase3b_result.manifest_df
p_qc = phase3b_result.channel_qc_df

print(f'Preprocessed manifest rows: {len(p_manifest)} (retained: {p_manifest["recording_retained_in_outputs"].sum()})')
print(f'Channel QC rows:            {len(p_qc)}')
display(p_manifest.head(15))

print('\n=== Exploratory Screening Flag Breakdown by Task Category (100% Retained) ===')
task_qc_summary = (
    p_manifest.groupby('task_category')
    .agg(
        n_total_recordings=('relative_source_path', 'count'),
        n_retained_in_outputs=('recording_retained_in_outputs', 'sum'),
        n_unflagged_allow_0=('is_unflagged_allow_0', 'sum'),
        n_unflagged_allow_1=('is_unflagged_allow_1', 'sum'),
        n_unflagged_allow_2=('is_unflagged_allow_2', 'sum'),
        n_total_low_std_channels=('n_low_std_channels', 'sum'),
        n_total_high_ptp_channels=('n_high_ptp_channels', 'sum'),
    )
    .reset_index()
)
display(task_qc_summary)

print('\n=== Sample Channel QC Rows with Exploratory Screening Flags ===')
display(
    p_qc[p_qc['screening_flag_any']][
        [
            'task_category',
            'relative_source_path',
            'channel_name',
            'post_filter_std_uV',
            'post_filter_ptp_uV',
            'post_filter_boundary_to_interior_std_ratio',
            'screening_flag_low_std',
            'screening_flag_high_ptp',
            'screening_flag_reasons',
            'screening_status',
        ]
    ].head(20)
)

## 9. Verify 1-to-1 Output Completeness Across All 690 Recordings and Confirm Source / Earlier-Phase Integrity

In [ ]:
verification_summary = verify_phase3b_outputs(
    output_dir=PHASE3B_OUTPUT_DIR,
    expected_manifest_df=manifest_df,
    preprocessing_config=config.preprocessing,
)
print('=== 1-to-1 Output Completeness & NPZ SHA-256 Verification ===')
print(json.dumps(verification_summary, indent=2))

raw_edf_hashes_after = snapshot_dir_hashes(DATA_ROOT)
phase1_hashes_after = snapshot_dir_hashes(PHASE1_AUDIT_DIR)
phase2_hashes_after = snapshot_dir_hashes(PHASE2_OUTPUT_DIR)
phase3a_hashes_after = snapshot_dir_hashes(PHASE3A_OUTPUT_DIR)

if raw_edf_hashes_before != raw_edf_hashes_after:
    raise RuntimeError('Integrity violation: Raw EDF files were modified!')
if phase1_hashes_before != phase1_hashes_after:
    raise RuntimeError('Integrity violation: Phase 1 audit files were modified!')
if phase2_hashes_before != phase2_hashes_after:
    raise RuntimeError('Integrity violation: Phase 2 signal-quality files were modified!')
if phase3a_hashes_before != phase3a_hashes_after:
    raise RuntimeError('Integrity violation: Phase 3A diagnostic files were modified!')

print('Confirmed: All raw EDF files, Phase 1 reports, Phase 2 reports, and Phase 3A reports are byte-for-byte unchanged.')

## 10. Display Phase 3B Report and Summary Figures

In [ ]:
from IPython.display import Image, Markdown

report_path = PHASE3B_OUTPUT_DIR / 'phase3b_preprocessing_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))

figures_dir = PHASE3B_OUTPUT_DIR / 'figures'
png_files = sorted(p for p in figures_dir.glob('*.png') if p.is_file() and p.stat().st_size > 0)
for fig_file in png_files:
    print(f'Figure: {fig_file}')
    display(Image(filename=str(fig_file)))